# Hand Tracking Model Benchmark

**Goal:** Determine the fastest, most reliable hand tracking model for a multi-person hack-and-slash game (Fruit Chop) running in Godot on laptop + mobile.

**What we're comparing:**
- **MediaPipe Hands** (current backend in Godot via GDMP)
- **YOLO26n-pose** (custom-trained on hand keypoints dataset, ONNX export)

**What we measure:**
- Inference latency (P50, P95, P99)
- FPS throughput
- Detection reliability during fast motion (dropout rate)
- Number of hands detected

**Runtimes tested (and why both matter):**
- **PyTorch + CUDA** -- theoretical ceiling of what the model can do on this GPU
- **ONNX Runtime** -- what we'd actually ship in a Godot GDExtension. The gap between these two reveals ONNX conversion overhead.

**Test conditions:**
- Two phases per model: **idle** (hands still) and **slash stress** (aggressive fast slashing)
- Camera: webcam at 1280x720
- Hardware: NVIDIA RTX 3070 Ti Laptop GPU, Intel CPU

**Research context:**
- [SOTA Survey](../docs/research/edge-hand-tracking-sota-2026.md)
- [Ecosystem Scan Round 2](../docs/research/ecosystem-scan-round2.md)
- [YOLO26 Training Plan](../docs/plan/yolo26-hand-tracking-pivot.md)

## 0. Setup

In [ ]:
import cv2
import time
import statistics
import numpy as np
from pathlib import Path

# Config
CAMERA_INDEX = 1        # 0 = built-in, 1 = NexiGo external
PHASE_DURATION = 15     # seconds per test phase
YOLO_MODEL_PT = '../runs/pose/hand-yolo26n/weights/best.pt'
YOLO_MODEL_ONNX = '../runs/pose/hand-yolo26n/weights/best.onnx'

print(f'YOLO PT exists: {Path(YOLO_MODEL_PT).exists()}')
print(f'YOLO ONNX exists: {Path(YOLO_MODEL_ONNX).exists()}')

# Verify camera
cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
ret, frame = cap.read()
print(f'Camera {CAMERA_INDEX}: {frame.shape if ret else "FAILED"}')
cap.release()

## 1. Export YOLO26 to ONNX (run once after training completes)

In [ ]:
from ultralytics import YOLO
import os

if not Path(YOLO_MODEL_ONNX).exists() and Path(YOLO_MODEL_PT).exists():
    model = YOLO(YOLO_MODEL_PT)
    onnx_path = model.export(format='onnx', imgsz=640, simplify=True)
    print(f'Exported: {onnx_path}')
    print(f'Size: {os.path.getsize(str(onnx_path)) / 1024 / 1024:.1f} MB')
elif Path(YOLO_MODEL_ONNX).exists():
    print(f'ONNX already exists: {YOLO_MODEL_ONNX}')
    print(f'Size: {os.path.getsize(YOLO_MODEL_ONNX) / 1024 / 1024:.1f} MB')
else:
    print('WARNING: No best.pt found. Is training finished?')

## 2. Print YOLO26 Training Results

In [ ]:
import csv

results_csv = '../runs/pose/hand-yolo26n/results.csv'
if Path(results_csv).exists():
    rows = list(csv.reader(open(results_csv)))
    header = rows[0]
    last = rows[-1]
    print(f'Epochs completed: {len(rows)-1}')
    print()
    for i, h in enumerate(header):
        h = h.strip()
        if 'map' in h.lower() or 'precision' in h.lower() or 'recall' in h.lower():
            print(f'  {h:>30s} = {float(last[i].strip()):.4f}')
else:
    print('No results.csv found yet')

## 3. Benchmark Helper Functions

In [ ]:
def run_benchmark(inference_fn, cap, duration, label):
    """Run a benchmark phase. inference_fn(frame) -> num_hands_detected."""
    print(f'  Running: {label} ({duration}s)...')
    print(f'  >>> INSTRUCTIONS: ', end='')
    if 'idle' in label.lower():
        print('Hold your hand(s) STILL in front of the camera.')
    else:
        print('SLASH your hands around aggressively! Fast, erratic motion.')
    
    # Countdown
    for i in range(3, 0, -1):
        print(f'  Starting in {i}...', end='\r')
        time.sleep(1)
    print(f'  GO! Recording for {duration}s...          ')
    
    latencies = []
    hands_per_frame = []
    start = time.time()
    
    while time.time() - start < duration:
        ret, frame = cap.read()
        if not ret:
            break
        
        t0 = time.perf_counter()
        num_hands = inference_fn(frame)
        t1 = time.perf_counter()
        
        latencies.append((t1 - t0) * 1000)
        hands_per_frame.append(num_hands)
    
    total_frames = len(latencies)
    dropout_frames = sum(1 for h in hands_per_frame if h == 0)
    
    return {
        'frames': total_frames,
        'fps': total_frames / duration,
        'latency_avg': statistics.mean(latencies),
        'latency_p50': statistics.median(latencies),
        'latency_p95': sorted(latencies)[int(len(latencies) * 0.95)] if latencies else 0,
        'latency_p99': sorted(latencies)[int(len(latencies) * 0.99)] if latencies else 0,
        'latency_min': min(latencies),
        'latency_max': max(latencies),
        'avg_hands': statistics.mean(hands_per_frame),
        'dropout_rate': dropout_frames / total_frames if total_frames else 1.0,
        'dropout_frames': dropout_frames,
    }

def print_results_table(results_dict):
    """Print a comparison table."""
    print(f"{'Model + Phase':<45} {'FPS':>5} {'P50ms':>6} {'P95ms':>6} {'Hands':>5} {'Drop%':>6}")
    print('-' * 75)
    for name, r in results_dict.items():
        print(f"{name:<45} {r['fps']:>5.1f} {r['latency_p50']:>6.1f} {r['latency_p95']:>6.1f} {r['avg_hands']:>5.1f} {r['dropout_rate']*100:>5.1f}%")

## 4. Benchmark: MediaPipe Hands

In [ ]:
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.4,
    min_tracking_confidence=0.3,
)

def mediapipe_infer(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)
    return len(result.multi_hand_landmarks) if result.multi_hand_landmarks else 0

# Warm up
cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
for _ in range(10):
    ret, frame = cap.read()
    mediapipe_infer(frame)

all_results = {}

print('=== MediaPipe Hands (CPU) ===')
all_results['MediaPipe / idle'] = run_benchmark(mediapipe_infer, cap, PHASE_DURATION, 'MediaPipe IDLE')
all_results['MediaPipe / slash stress'] = run_benchmark(mediapipe_infer, cap, PHASE_DURATION, 'MediaPipe SLASH STRESS')

cap.release()
hands.close()

print_results_table(all_results)

## 5. Benchmark: YOLO26n-pose (via Ultralytics, GPU)

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO(YOLO_MODEL_PT)

def yolo_infer(frame):
    results = yolo_model(frame, imgsz=640, verbose=False, conf=0.4)
    return len(results[0].boxes) if results[0].boxes is not None else 0

# Warm up
cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
for _ in range(10):
    ret, frame = cap.read()
    yolo_infer(frame)

print('=== YOLO26n-pose (PyTorch + CUDA) ===')
all_results['YOLO26 PyTorch+CUDA / idle'] = run_benchmark(yolo_infer, cap, PHASE_DURATION, 'YOLO26 IDLE')
all_results['YOLO26 PyTorch+CUDA / slash stress'] = run_benchmark(yolo_infer, cap, PHASE_DURATION, 'YOLO26 SLASH STRESS')

cap.release()

print_results_table(all_results)

## 6. Benchmark: YOLO26n-pose (ONNX Runtime — simulates Godot deployment)

In [ ]:
import onnxruntime as ort

print(f'ONNX Runtime version: {ort.__version__}')
print(f'Available providers: {ort.get_available_providers()}')

if not Path(YOLO_MODEL_ONNX).exists():
    print('ONNX model not found. Run the export cell above first.')
else:
    session = ort.InferenceSession(YOLO_MODEL_ONNX, providers=ort.get_available_providers())
    input_name = session.get_inputs()[0].name
    input_shape = session.get_inputs()[0].shape
    print(f'Input: {input_name}, shape: {input_shape}')
    
    h, w = input_shape[2], input_shape[3]
    
    def yolo_onnx_infer(frame):
        resized = cv2.resize(frame, (w, h))
        blob = resized.astype(np.float32) / 255.0
        blob = np.transpose(blob, (2, 0, 1))[np.newaxis]
        outputs = session.run(None, {input_name: blob})
        # Post-processing would extract hand count here.
        # For latency benchmark, raw inference time is what matters.
        return -1  # Can't count hands without NMS post-processing
    
    # Warm up
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    for _ in range(10):
        ret, frame = cap.read()
        yolo_onnx_infer(frame)
    
    print('=== YOLO26n-pose (ONNX Runtime) ===')
    all_results['YOLO26 ONNX / idle'] = run_benchmark(yolo_onnx_infer, cap, PHASE_DURATION, 'YOLO26 ONNX IDLE')
    all_results['YOLO26 ONNX / slash stress'] = run_benchmark(yolo_onnx_infer, cap, PHASE_DURATION, 'YOLO26 ONNX SLASH STRESS')
    
    cap.release()

## 7. Final Comparison

In [ ]:
print('=' * 80)
print('  FINAL COMPARISON: Hand Tracking Models for Hack-and-Slash Games')
print('=' * 80)
print()
print('Key question: Which model gives lowest latency during FAST SLASHING MOTION?')
print('Dropout rate = % of frames where NO hands were detected (lower = better)')
print()
print_results_table(all_results)
print()
print('NOTES:')
print('- MediaPipe runs on CPU only (no GPU delegate on Windows)')
print('- YOLO26 PyTorch uses CUDA GPU — shows theoretical best performance')
print('- YOLO26 ONNX is what we would ship in a Godot GDExtension')
print('- For iOS deployment, Apple Vision framework is another option (native hand tracking)')
print()
print('DECISION CRITERIA:')
print('- If YOLO26 ONNX slash P50 < MediaPipe slash P50: switch to YOLO26')
print('- If dropout rate is lower: that model is more reliable during fast motion')
print('- On mobile: YOLO26 can use GPU via DirectML/CoreML/NNAPI; MediaPipe uses GPU delegate')

## 8. Next Steps

Based on results above:

**If YOLO26 wins:**
- Build ONNX Runtime GDExtension for Godot
- Reference: [HandLocator C++ code](https://github.com/Sputnikboi/HandLocator)
- Export to mobile: `model.export(format='litert', quantize=True)` for Android

**If MediaPipe wins:**
- Keep current GDMP pipeline
- Focus on the latency optimizations already shipped (downscale, throttle, 1-Euro)
- Consider RTMPose as future upgrade (needs separate mmpose setup)

**Either way:**
- Evaluate [RTMPose Hand5 model](../models/rtmpose/) when mmpose venv is set up
- Build [gesture classifier](../docs/plan/gesture-classifier.md) for slash direction detection
- Test [Apple Vision framework](https://www.createwithswift.com/detecting-hand-pose-with-the-vision-framework/) for iOS-native hand tracking